## SAT-style Question Answering with GGUF Models

Learning objective: Load a small model from a shared directory and use it to answer SAT-style questions with a small language model.

## What this notebook teaches
- How to locate a shared folder of GGUF weights on your laptop or hub
- How to load a model with llama-cpp-python and walk through a tutor-style prompt
- How to inspect the model's answer and reasoning with reflection checkpoints
- How to compare the model's reasoning across random, filtered, and batched SAT items

## Where the questions come from
- We download the PineSAT Questionbank API  at https://pinesat.com/api/questions.
- PineSAT hosts community-built SAT-style questions so you can practice with authentic formats without licensing hurdles.
- The endpoint replies in JSON (JavaScript Object Notation, a plain text key-value format) so we can inspect passages, choices, and answer keys with simple loops.

## How to navigate the lesson
1. Every markdown cell previews the next code cell so you always know why a command matters.
2. Reflection checkpoints append your thoughts to answers.txt so you can track how your understanding changes.
3. Later cells batch four easy questions into a table so you can see accuracy trends without extra plotting.

Tip: Keep an eye on the model context size (the amount of text it can read at once) and thread count so you do not overload a shared machine.

In [38]:
# ── imports ──────────────────────────────────────────────────────────────────
from llama_cpp import Llama  # loads llama-cpp-python so we can run GGUF models
import os                    # lets us work with file paths
import random                # lets us pick random items from a list
import requests              # lets us call web APIs over HTTP
import pandas as pd          # helps with working with tables and spreadsheets
import re                    # lets us search text with patterns
import json                  # lets us parse structured JSON output from the model
import ipywidgets as widgets         # adds dropdown menus and buttons for simple UI
from IPython.display import display, clear_output, HTML

print("✅ All packages ready.")

✅ All packages ready.


### Step 1: Find the model directory

Before loading a model, we need to know where the `.gguf` files live.
This varies by machine:
- **JupyterHub**: usually `/home/jovyan/shared/` or `/home/jovyan/shared_readwrite/`
- **Laptop**: wherever you downloaded the model files

The cell below checks each possible path and picks the first one that exists.

In [2]:
# list of directories to check — add your own path here if needed
possible_directories = [
    "/home/jovyan/shared/",           # JupyterHub shared folder
    "/home/jovyan/shared_readwrite/", # JupyterHub read-write shared folder
    "/Users/ericvandusen/SmallLM/Models"  # local laptop path
]

# check each path and keep only the ones that actually exist on this machine
existing_directories = []
for directory_path in possible_directories:
    if os.path.exists(directory_path):
        print("Found possible directory:", directory_path)
        existing_directories.append(directory_path)
    else:
        print("Did not find:", directory_path)

Found possible directory: /home/jovyan/shared/
Did not find: /home/jovyan/shared_readwrite/
Did not find: /Users/ericvandusen/SmallLM/Models


### Step 2: Select the model directory

The cell below automatically picks the first directory that was found.
If you want to use a different path, you can type it in when prompted.
If no directory was found, you will be asked to enter a path manually.

In [3]:
# use the first directory that was found above
# if none were found, ask the user to type a path manually
if len(existing_directories) > 0:
    model_directory = existing_directories[0]
    print("Using this directory by default:", model_directory)
else:
    model_directory = input("Type a directory path that contains your .gguf files: ")

print("Current model directory:", model_directory)

Using this directory by default: /home/jovyan/shared/
Current model directory: /home/jovyan/shared/


### Step 3: Choose a model

The cell below scans the directory for `.gguf` files and shows them in a dropdown menu.
If a Qwen model is found, it will be selected by default.

**Important:** Make your selection in the dropdown before running the next cell to load the model.

In [4]:
# scan the directory and collect all .gguf filenames
available_models = []
for filename in os.listdir(model_directory):
    if filename.endswith(".gguf"):
        available_models.append(filename)

if len(available_models) == 0:
    print("No .gguf files found in", model_directory)
else:
    print("Models found in", model_directory)

    # default to the first model found
    # if a Qwen model exists, prefer it over others
    dropdown_default = available_models[0]
    for candidate_name in available_models:
        lowercase_name = candidate_name.lower()
        if "qwen" in lowercase_name:
            dropdown_default = candidate_name
            break

    model_dropdown = widgets.Dropdown(
        options=available_models,
        description="Model:",
        value=dropdown_default
    )
    display(model_dropdown)
    print("Use the dropdown to pick a model, then run the next cell to load it.")

Models found in /home/jovyan/shared/


Dropdown(description='Model:', options=('qwen2-1_5b-instruct-q4_0.gguf', 'tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf…

Use the dropdown to pick a model, then run the next cell to load it.


### Checkpoint #1: Model Card Research

Look up the model card for the model you selected and answer these questions:
- What was this model trained on?
- What is its maximum context size?
- How many threads does it use by default?

You can find model cards on [Hugging Face](https://huggingface.co) by searching the model name.

### Step 4: Load the model

Now we load the model you selected into memory using `llama-cpp-python`.

- `n_ctx=4096` — the model can read up to 4,096 tokens at once
- `n_threads=4` — uses 4 CPU threads to run inference
- `chat_format="chatml"` — tells the model how to format conversations (required for Qwen)
- `verbose=False` — suppresses internal logs to keep the output clean

This may take 10–30 seconds depending on the model size and your machine.

In [5]:
# read the selected model name from the dropdown
# and build the full file path by joining the directory and filename
selected_model_name = model_dropdown.value
model_path = os.path.join(model_directory, selected_model_name)

print(f"Selected: {selected_model_name}")
print(f"Full path: {model_path}")
print(f"File exists: {os.path.exists(model_path)}")

# load the model into memory
print("⏳ Loading model...")
model = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=4,
    chat_format="chatml",  # required for Qwen — controls how messages are formatted
    verbose=False
)
print(f"✅ Loaded: {selected_model_name}")


Selected: qwen2-1_5b-instruct-q4_0.gguf
Full path: /home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf
File exists: True
⏳ Loading model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Loaded: qwen2-1_5b-instruct-q4_0.gguf


In [6]:
try:
    from huggingface_hub import ModelCard
except ImportError:
    %pip install huggingface_hub
    from huggingface_hub import ModelCard

In [8]:
print("""
HuggingFace model ID is NOT the same as the filename.

  Filename  : qwen2-1_5b-instruct-q4_0.gguf   ← this is the local file
  Model ID  : Qwen/Qwen2-1.5B-Instruct         ← this is what you need

The model ID follows the format:  OrganizationName/ModelName
You can find it in the URL on HuggingFace, e.g.:
  https://huggingface.co/Qwen/Qwen2-1.5B-Instruct
                          ↑ this part is the model ID
""")

hf_id = input("Enter the HuggingFace model ID (e.g. Qwen/Qwen2-1.5B-Instruct): ")

try:
    model_card = ModelCard.load(hf_id)
    print(model_card.text[:3000])
except Exception as e:
    print("Could not load model card:", e)
    print("Check that the model ID is correct at https://huggingface.co/" + hf_id)


HuggingFace model ID is NOT the same as the filename.

  Filename  : qwen2-1_5b-instruct-q4_0.gguf   ← this is the local file
  Model ID  : Qwen/Qwen2-1.5B-Instruct         ← this is what you need

The model ID follows the format:  OrganizationName/ModelName
You can find it in the URL on HuggingFace, e.g.:
  https://huggingface.co/Qwen/Qwen2-1.5B-Instruct
                          ↑ this part is the model ID



Enter the HuggingFace model ID (e.g. Qwen/Qwen2-1.5B-Instruct):  Qwen/Qwen2-1.5B-Instruct


README.md: 0.00B [00:00, ?B/s]


# Qwen2-1.5B-Instruct

## Introduction

Qwen2 is the new series of Qwen large language models. For Qwen2, we release a number of base language models and instruction-tuned language models ranging from 0.5 to 72 billion parameters, including a Mixture-of-Experts model. This repo contains the instruction-tuned 1.5B Qwen2 model.

Compared with the state-of-the-art opensource language models, including the previous released Qwen1.5, Qwen2 has generally surpassed most opensource models and demonstrated competitiveness against proprietary models across a series of benchmarks targeting for language understanding, language generation, multilingual capability, coding, mathematics, reasoning, etc.

For more details, please refer to our [blog](https://qwenlm.github.io/blog/qwen2/), [GitHub](https://github.com/QwenLM/Qwen2), and [Documentation](https://qwen.readthedocs.io/en/latest/).
<br>

## Model Details
Qwen2 is a language model series including decoder language models of different model size

### Warm-up: Test the model with a simple question

Before pulling real SAT questions, we verify the model is working correctly
with a hand-written algebra question.

The model is asked to return a **structured JSON answer** instead of free-form text:
```json
{"choice": "B"}
```

This makes grading reliable — we always know exactly where to find the answer.
The prompt includes:
- a simple algebraic equation to solve
- four answer choices (A, B, C, D)
- a strict instruction to return only one JSON field named `choice`

In [9]:
# ── question setup ────────────────────────────────────────────────────────────
warmup_question = "If 3x + 5 = 14, what is the value of x?"
warmup_choice_map = {
    "A": "2",
    "B": "3",  # correct answer
    "C": "4",
    "D": "5"
}

# format choices as "A: 2\nB: 3\n..." for the prompt
warmup_choices_text = "\n".join(
    f"{label}: {value}" for label, value in warmup_choice_map.items()
)

# ── build the prompt ──────────────────────────────────────────────────────────
warmup_prompt = f"""Here is an SAT-style multiple-choice question:

Question: {warmup_question}

Choices:
{warmup_choices_text}

Respond ONLY with valid JSON in this exact format: {{"choice": "B"}}
Choose one letter from A, B, C, or D."""

# system message tells the model its role and output format
# user message contains the actual question
messages = [
    {
        "role": "system",
        "content": (
            "You are an SAT math solver. "
            "Always respond with valid JSON containing a single field 'choice' "
            "set to one of: A, B, C, or D. No explanation, no extra text."
        )
    },
    {"role": "user", "content": warmup_prompt}
]

# ── call the model ────────────────────────────────────────────────────────────
# response_format enforces JSON output with a schema — the model must return
# a valid JSON object with a "choice" field set to one of A, B, C, D
warmup_response = model.create_chat_completion(
    messages=messages,
    response_format={
        "type": "json_object",
        "schema": {
            "type": "object",
            "properties": {
                "choice": {
                    "type": "string",
                    "enum": ["A", "B", "C", "D"]
                }
            },
            "required": ["choice"]
        }
    },
    temperature=0.0  # no randomness — always pick the most likely answer
)

warmup_raw_text = warmup_response["choices"][0]["message"]["content"].strip()
print("Raw model output:", warmup_raw_text)

# ── parse the response ────────────────────────────────────────────────────────
# try JSON first; fall back to regex if parsing fails
warmup_choice = ""
try:
    warmup_json = json.loads(warmup_raw_text)
    warmup_choice = str(warmup_json.get("choice", "")).strip().upper()
except json.JSONDecodeError as parse_error:
    print("JSON parse failed:", parse_error)
    # fallback: scan the raw text for a lone A/B/C/D
    fallback_match = re.search(r'\b([A-D])\b', warmup_raw_text.upper())
    if fallback_match:
        warmup_choice = fallback_match.group(1)

if warmup_choice not in ["A", "B", "C", "D"]:
    print("WARNING: Could not extract a valid choice from model output.")
    warmup_choice = "UNKNOWN"

# ── check the answer ──────────────────────────────────────────────────────────
correct = "B"
print(f"Parsed choice:  {warmup_choice}")
print(f"Correct choice: {correct}")
print(f"Result: {'✓ Correct' if warmup_choice == correct else '✗ Wrong'}")

Raw model output: {"choice": "B"}
Parsed choice:  B
Correct choice: B
Result: ✓ Correct


## Source an open source set of SAT-style questions

Download the SAT Questionbank from PineSAT to pull authentic practice questions.
We will use the API endpoint at `https://pinesat.com/api/questions`, which returns
a JSON object with passages, questions, choices, and answer keys.

The endpoint supports two sections:
- `section=english` — reading and writing questions
- `section=math` — algebra and data analysis questions

We can loop through this data to inspect the format and content of the questions.

In [10]:
# fetch SAT-style questions from the PineSAT API
# — returns two lists: one for English, one for Math
base_url = "https://pinesat.com/api/questions"

# fetch questions from both sections — this requires an internet connection
try:
    english_questions = requests.get(base_url, params={"section": "english"}).json()
    math_questions    = requests.get(base_url, params={"section": "math"}).json()
except Exception as e:
    print("API request failed:", e)
    print("Check your internet connection or try again later.")
    raise

# wrap in a DataFrame so we can inspect the structure easily
english_nested = pd.DataFrame(english_questions)
math_nested    = pd.DataFrame(math_questions)

print("English questions:", len(english_nested))
print("Math questions:",    len(math_nested))

English questions: 1443
Math questions: 1031


In [12]:
english_questions[0]

{'id': 'random_id_a1',
 'domain': 'Information and Ideas',
 'visuals': {'type': 'null', 'svg_content': 'null'},
 'question': {'choices': {'A': 'Suppressing opinions robs future generations of the chance to hear them, even if they disagree with them.',
   'B': 'It is harmful to silence opinions that are held by a majority of people.',
   'C': 'People who dissent from an opinion are more likely to be harmed by its suppression than those who hold it.',
   'D': 'It is important to respect all opinions, even if they are wrong.'},
  'question': 'What is Mill\'s main point in this passage from "On Liberty"?',
  'paragraph': 'In the essay "On Liberty," John Stuart Mill argues that "the peculiar evil of silencing the expression of an opinion is, that it is robbing the human race; posterity as well as the existing generation; those who dissent from the opinion, still more than those who hold it."  What is Mill\'s main point in this passage?',
  'explanation': 'Mill argues that suppressing opinio

In [13]:
english_questions[0]["question"]

{'choices': {'A': 'Suppressing opinions robs future generations of the chance to hear them, even if they disagree with them.',
  'B': 'It is harmful to silence opinions that are held by a majority of people.',
  'C': 'People who dissent from an opinion are more likely to be harmed by its suppression than those who hold it.',
  'D': 'It is important to respect all opinions, even if they are wrong.'},
 'question': 'What is Mill\'s main point in this passage from "On Liberty"?',
 'paragraph': 'In the essay "On Liberty," John Stuart Mill argues that "the peculiar evil of silencing the expression of an opinion is, that it is robbing the human race; posterity as well as the existing generation; those who dissent from the opinion, still more than those who hold it."  What is Mill\'s main point in this passage?',
 'explanation': 'Mill argues that suppressing opinions is a harm to everyone, including future generations, because it prevents them from hearing and potentially engaging with these i

In [14]:
english_questions[0]["question"]['question']

'What is Mill\'s main point in this passage from "On Liberty"?'

In [15]:
english_questions[0]["question"]['choices']

{'A': 'Suppressing opinions robs future generations of the chance to hear them, even if they disagree with them.',
 'B': 'It is harmful to silence opinions that are held by a majority of people.',
 'C': 'People who dissent from an opinion are more likely to be harmed by its suppression than those who hold it.',
 'D': 'It is important to respect all opinions, even if they are wrong.'}

In [16]:
english_questions[0]["question"]['paragraph']

'In the essay "On Liberty," John Stuart Mill argues that "the peculiar evil of silencing the expression of an opinion is, that it is robbing the human race; posterity as well as the existing generation; those who dissent from the opinion, still more than those who hold it."  What is Mill\'s main point in this passage?'

In [17]:
english_questions[0]["question"]['correct_answer']

'A'

In [19]:
# explore the structure of a single question from the API
# the data is nested: each entry has a "question" object with several fields
sample = english_questions[0]["question"]

print("Paragraph:")
print(sample.get("paragraph", "(none)"))
print()
print("Question :", sample["question"])
print()
print("Choices  :")
for label, text in sample["choices"].items():
    print(f"  {label}: {text}")
print()
print("Answer   :", sample["correct_answer"])

Paragraph:
In the essay "On Liberty," John Stuart Mill argues that "the peculiar evil of silencing the expression of an opinion is, that it is robbing the human race; posterity as well as the existing generation; those who dissent from the opinion, still more than those who hold it."  What is Mill's main point in this passage?

Question : What is Mill's main point in this passage from "On Liberty"?

Choices  :
  A: Suppressing opinions robs future generations of the chance to hear them, even if they disagree with them.
  B: It is harmful to silence opinions that are held by a majority of people.
  C: People who dissent from an opinion are more likely to be harmed by its suppression than those who hold it.
  D: It is important to respect all opinions, even if they are wrong.

Answer   : A


### Inspect a random question

Now let's look at a randomly selected question to see how the data varies across the bank.
Unlike the previous cell which always showed the first question, this one picks a different
question each time you run it — so you can get a feel for the range of topics and formats.

In [23]:
# explore the structure of a single math question
sample = random.choice(math_questions)["question"]

# replace LaTeX \pi with the actual symbol for cleaner output
def clean_text(text):
    return str(text).replace(r"\pi", "π").replace(r"\times", "×").replace(r"\div", "÷").replace("$", "")

print("Question :", clean_text(sample["question"]))
print()
print("Choices  :")
for label, text in sample["choices"].items():
    print(f"  {label}: {clean_text(text)}")
print()
print("Answer   :", sample["correct_answer"])


Question : The vertices of a triangle are located at the points (0,0), (4,0), and (2,3). What is the area of the triangle?

Choices  :
  A: 6
  B: 12
  C: 18
  D: 24

Answer   : A


### Ask the model — English

Now we pass a randomly selected English question to the model using the same
JSON-format prompt from the warm-up. This lets you compare how the model handles
a real SAT reading question versus the simple algebra warm-up.

In [24]:
# ── pick a random English question ───────────────────────────────────────────
random_entry        = random.choice(english_questions)
random_question_text  = random_entry["question"]["question"]
random_paragraph_text = random_entry["question"].get("paragraph", "")
random_choices_raw    = random_entry["question"]["choices"]
correct_answer        = random_entry["question"].get("correct_answer", "")

# ── normalise choices into a dict {A: text, B: text, ...} ────────────────────
# choices can arrive as a dict or a list depending on the question
choice_labels  = ["A", "B", "C", "D", "E", "F"]
random_choice_map = {}

if isinstance(random_choices_raw, dict):
    for raw_label in random_choices_raw:
        clean_label = str(raw_label).strip().upper()
        random_choice_map[clean_label] = str(random_choices_raw[raw_label])
elif isinstance(random_choices_raw, list):
    for item_index in range(len(random_choices_raw)):
        if item_index < len(choice_labels):
            random_choice_map[choice_labels[item_index]] = str(random_choices_raw[item_index])
else:
    random_choice_map["A"] = str(random_choices_raw)

valid_labels      = [l for l in choice_labels if l in random_choice_map]
random_option_lines = [label + ": " + random_choice_map[label] for label in valid_labels]
random_options_text = "\n".join(random_option_lines)

# ── build the prompt ──────────────────────────────────────────────────────────
# include the passage only if one exists (not all questions have a paragraph)
if random_paragraph_text:
    english_prompt_question_text = "Passage: " + random_paragraph_text + "\n\nQuestion: " + random_question_text
else:
    english_prompt_question_text = "Question: " + random_question_text

english_structured_prompt = """
{question_text}

Options:
{options_text}

Return your answer as JSON with this exact shape:
{{"choice": "A"}}
Use only one label from the provided options.
"""

# ── print the question so we can read it before seeing the model's answer ─────
print("Paragraph:", random_paragraph_text if random_paragraph_text else "(none)")
print()
print("Question :", random_question_text)
print()
print("Options  :")
for option_line in random_option_lines:
    print(" ", option_line)
print()
print("Answer   :", correct_answer)

# ── call the model ────────────────────────────────────────────────────────────
random_messages = [
    {"role": "system", "content": "You are a helpful tutor. Return only valid JSON with one field named choice."},
    {"role": "user",   "content": english_structured_prompt.format(
        question_text=english_prompt_question_text,
        options_text=random_options_text
    )}
]

random_response = model.create_chat_completion(
    messages=random_messages,
    response_format={
        "type": "json_object",
        "schema": {
            "type": "object",
            "properties": {
                "choice": {"type": "string", "enum": valid_labels}
            },
            "required": ["choice"]
        }
    },
    temperature=0.0
)

# ── parse the response ────────────────────────────────────────────────────────
random_raw_text = random_response["choices"][0]["message"]["content"]
random_choice   = ""

try:
    random_json   = json.loads(random_raw_text)
    random_choice = str(random_json.get("choice", "")).strip().upper()
except Exception as parse_error:
    print("Could not parse JSON directly:", parse_error)
    fallback_match = re.search(r"\b(" + "|".join(valid_labels) + r")\b", random_raw_text.upper())
    if fallback_match:
        random_choice = fallback_match.group(1)

# ── print result ──────────────────────────────────────────────────────────────
print("Raw model output:", random_raw_text)
print()
print(f"Model choice  : {random_choice}")
print(f"Correct answer: {correct_answer}")
print(f"Result        : {'✓ Correct' if random_choice == correct_answer else '✗ Wrong'}")

Paragraph: The author of the passage is a prominent philosopher who is writing a book on the nature of reality. The book is intended for a general audience, so the author tries to explain complex ideas in a clear and concise way. However, the author also wants to maintain a sense of intellectual rigor and avoid oversimplification. What is the most likely reason why the author would use the word "however" in the passage?

Question : In this passage, the word "however" is used to

Options  :
  A: introduce a contrasting idea.
  B: emphasize the importance of the author’s message.
  C: describe the author’s feelings and experiences.
  D: suggest that the author is addressing a large audience.

Answer   : A
Raw model output: {"choice": "A"}

Model choice  : A
Correct answer: A
Result        : ✓ Correct


### Ask the model — Math

Now we repeat the same process with a randomly selected Math question.
Compare the model's performance across the two subjects — does it do better on
Math or English? Does the question format affect the result?

In [25]:
# pick a random math question and ask the model to answer it
random_entry = random.choice(math_questions)

# extract question text and choices
random_question_text = random_entry["question"]["question"]
random_choices_raw   = random_entry["question"]["choices"]

# normalize choices into a dict: {"A": "...", "B": "...", ...}
# the API sometimes returns a dict, sometimes a list
choice_labels    = ["A", "B", "C", "D", "E", "F"]
random_choice_map = {}

if isinstance(random_choices_raw, dict):
    for raw_label in random_choices_raw:
        clean_label = str(raw_label).strip().upper()
        random_choice_map[clean_label] = str(random_choices_raw[raw_label])
elif isinstance(random_choices_raw, list):
    for item_index in range(len(random_choices_raw)):
        if item_index < len(choice_labels):
            random_choice_map[choice_labels[item_index]] = str(random_choices_raw[item_index])
else:
    random_choice_map["A"] = str(random_choices_raw)

# collect only the labels that actually exist in this question
valid_labels = [label for label in choice_labels if label in random_choice_map]

random_option_lines = [label + ": " + random_choice_map[label] for label in valid_labels]
random_options_text = "\n".join(random_option_lines)

# ── build the prompt ──────────────────────────────────────────────────────────
math_structured_prompt = """
Question: {question_text}

Options:
{options_text}

Return your answer as JSON with this exact shape:
{{"choice": "A"}}
Use only one label from the provided options.
"""

print("Question:", random_question_text)
print("Options:")
for option_line in random_option_lines:
    print(option_line)

# ── call the model ────────────────────────────────────────────────────────────
random_messages = [
    {"role": "system", "content": "You are a helpful tutor. Return only valid JSON with one field named choice."},
    {"role": "user",   "content": math_structured_prompt.format(question_text=random_question_text, options_text=random_options_text)}
]

random_response = model.create_chat_completion(
    messages=random_messages,
    response_format={
        "type": "json_object",
        "schema": {
            "type": "object",
            "properties": {
                "choice": {"type": "string", "enum": valid_labels}
            },
            "required": ["choice"]
        }
    },
    temperature=0.0
)

# ── parse the response ────────────────────────────────────────────────────────
random_raw_text = random_response["choices"][0]["message"]["content"]
print("Raw model output:", random_raw_text)

random_choice = ""
try:
    random_json   = json.loads(random_raw_text)
    random_choice = str(random_json.get("choice", "")).strip().upper()
except Exception as parse_error:
    print("Could not parse JSON directly:", parse_error)

# fallback: scan raw text for a valid label if JSON parsing failed
if random_choice not in valid_labels:
    fallback_pattern = r"\b(" + "|".join(valid_labels) + r")\b"
    fallback_match   = re.search(fallback_pattern, random_raw_text.upper())
    if fallback_match is not None:
        random_choice = fallback_match.group(1)

print("Parsed choice:", random_choice)

Question: The equation $(x+2)^2 + (y-3)^2 = 25$ represents a circle in the xy-plane. What is the radius of the circle? 
Options:
A: 2
B: 5
C: 10
D: 25
Raw model output: {"choice": "A"}
Parsed choice: A


### Checkpoint #2: Reflection

Why are we testing the model with questions from both English and Math domains?
Think about what it means to evaluate a model across different subjects.

In [29]:
# student selects a reason and the answer is saved to answers.txt
reason_widget = widgets.RadioButtons(
    options=[
        "A: To see if the model performs equally well across different subjects",
        "B: To test whether the model can handle different question formats and reasoning types",
        "C: Because SAT has two sections so we should test both",
        "D: All of the above"
    ],
    description="Your answer:",
    layout=widgets.Layout(width="600px")
)
submit_btn = widgets.Button(description="Submit", button_style="success")
out = widgets.Output()

def on_submit(_):
    with out:
        clear_output()
        selected = reason_widget.value[0]  # get the letter A/B/C/D
        if selected == "D":
            print("✓ Correct! All three reasons are valid.")
        else:
            print(f"Partially right, but D (All of the above) is the most complete answer.")
        with open('answers.txt', 'a') as f:
            f.write("Checkpoint 2: " + reason_widget.value + "\n")

submit_btn.on_click(on_submit)
display(reason_widget, submit_btn, out)

RadioButtons(description='Your answer:', layout=Layout(width='600px'), options=('A: To see if the model perfor…

Button(button_style='success', description='Submit', style=ButtonStyle())

Output()

### Build a mini SAT practice set

So far we tested the model one question at a time. Now we build a small batch
so we can see how it performs across multiple questions at once.

Use the dropdowns below to filter by difficulty, subject, and number of questions.
The model will answer each one and we'll collect the results in a table.

In [30]:
# dropdowns for filtering the question bank
# student picks difficulty, subject, and number of questions before running the next cell
difficulty_widget = widgets.Dropdown(
    options=["Easy", "Medium", "Hard"],
    value="Easy",
    description="Difficulty:"
)
section_widget = widgets.Dropdown(
    options=["English", "Math"],
    value="English",
    description="Section:"
)
num_questions_widget = widgets.Dropdown(
    options=[2, 4, 8, 10, 12],
    value=4,
    description="# Questions:"
)

display(difficulty_widget)
display(section_widget)
display(num_questions_widget)
print("Choose your settings above, then run the next cell to test the model.")

Dropdown(description='Difficulty:', options=('Easy', 'Medium', 'Hard'), value='Easy')

Dropdown(description='Section:', options=('English', 'Math'), value='English')

Dropdown(description='# Questions:', index=1, options=(2, 4, 8, 10, 12), value=4)

Choose your settings above, then run the next cell to test the model.


Now send each question to the model and collect the results in a table.

In [40]:
# read the widget values set by the student
chosen_difficulty = difficulty_widget.value
chosen_section    = section_widget.value
chosen_count      = num_questions_widget.value

# pick the right question pool based on the selected section
question_pool = english_questions if chosen_section == "English" else math_questions

# filter by difficulty and shuffle so we get a fresh sample each run
filtered_questions = [
    q for q in question_pool
    if q.get("difficulty", "").lower() == chosen_difficulty.lower()
]
random.shuffle(filtered_questions)
practice_set = filtered_questions[:chosen_count]

# clean LaTeX formatting from text — removes \(...\), $...$, and common symbols
def clean_text(text):
    text = str(text)
    text = text.replace(r"\(", "").replace(r"\)", "")
    text = text.replace(r"\[", "").replace(r"\]", "")
    text = text.replace(r"\pi", "π").replace(r"\times", "×").replace(r"\div", "÷")
    text = text.replace(r"\neq", "≠").replace(r"\leq", "≤").replace(r"\geq", "≥")  # 加這行
    text = re.sub(r"\$([^$]+)\$", r"\1", text)
    return text.strip()

# unpack each question into separate lists for easy access later
practice_questions  = [clean_text(q["question"]["question"])           for q in practice_set]
practice_choices    = [q["question"]["choices"]                        for q in practice_set]
practice_paragraphs = [
    "" if q["question"].get("paragraph") in (None, "null", "")
    else clean_text(q["question"].get("paragraph", ""))
    for q in practice_set
]
practice_answers    = [q["question"].get("correct_answer", "") for q in practice_set]

print(f"Built a practice set of {len(practice_questions)} {chosen_difficulty} {chosen_section} questions.\n")

# print all questions first
for i in range(len(practice_questions)):
    print(f"Q{i+1}: {practice_questions[i]}")
    if practice_paragraphs[i]:
        print("Passage:", practice_paragraphs[i])
    print("Choices:", {k: clean_text(v) for k, v in practice_choices[i].items()})
    print()

# answer key hidden in a collapsible HTML details block
answer_rows = "".join(
    f"<tr><td style='padding:6px 14px;'>Q{i+1}</td>"
    f"<td style='padding:6px 14px; font-weight:bold;'>{practice_answers[i]}</td></tr>"
    for i in range(len(practice_answers))
)
display(HTML(f"""
<details>
  <summary style="cursor:pointer; font-family:monospace; font-size:13px;
                  padding:8px 12px; background:#fafbfc; border:1px solid #d0d7de;
                  border-radius:8px; display:inline-block;">
    ▶ Show answer key
  </summary>
  <table style="border-collapse:collapse; font-family:monospace; font-size:13px; margin-top:8px;">
    <tr style="background:#e9ecef;">
      <th style="padding:6px 14px; text-align:left;">Question</th>
      <th style="padding:6px 14px; text-align:left;">Correct Answer</th>
    </tr>
    {answer_rows}
  </table>
</details>
"""))

Built a practice set of 4 Medium Math questions.

Q1: Which of the following is equivalent to (x+y)^2 - (x-y)^2?
Choices: {'A': '2x^2 + 2y^2', 'B': '4xy', 'C': '2x^2 - 2y^2', 'D': 'x^2 - y^2'}

Q2: If 2x + 3y = 12 and x - y = 1, what is the value of x?
Choices: {'A': '2', 'B': '3', 'C': '4', 'D': '5'}

Q3: A circle has a radius of 5 units. What is the circumference of the circle?
Choices: {'A': '5π', 'B': '10π', 'C': '25π', 'D': '50π'}

Q4: A circle has a radius of 5 units.  What is the circumference of the circle, in units?  (Express your answer in terms of π.)
Choices: {'A': '5π', 'B': '10π', 'C': '25π', 'D': '50π'}



Question,Correct Answer
Q1,B
Q2,B
Q3,B
Q4,B


### Run the batch — ask the model every question

The cell below sends each question to the model one at a time and collects the results.
For each question it:
1. Builds a prompt with the passage (if any), question, and choices
2. Asks the model to return a JSON answer with a single `choice` field
3. Compares the model's answer to the correct answer key
4. Stores the result in a table

This may take a minute depending on how many questions you selected.

In [41]:
batch_results = []
choice_labels = ["A", "B", "C", "D", "E", "F"]

for item_index in range(len(practice_questions)):

    # build the full question text — include passage if one exists
    passage_text = practice_paragraphs[item_index]
    if len(passage_text) > 0:
        full_question_text = "Passage: " + passage_text + "\n\nQuestion: " + practice_questions[item_index]
    else:
        full_question_text = "Question: " + practice_questions[item_index]

    # normalize choices into a dict: {"A": "...", "B": "...", ...}
    # API sometimes returns a dict, sometimes a list
    question_choices_raw = practice_choices[item_index]
    question_choice_map  = {}

    if isinstance(question_choices_raw, dict):
        for raw_label in question_choices_raw:
            clean_label = str(raw_label).strip().upper()
            question_choice_map[clean_label] = clean_text(str(question_choices_raw[raw_label]))
    elif isinstance(question_choices_raw, list):
        for choice_index in range(len(question_choices_raw)):
            if choice_index < len(choice_labels):
                question_choice_map[choice_labels[choice_index]] = clean_text(str(question_choices_raw[choice_index]))
    else:
        question_choice_map["A"] = clean_text(str(question_choices_raw))

    # collect only labels that exist in this question
    valid_labels = [label for label in choice_labels if label in question_choice_map]
    options_text = "\n".join(label + ": " + question_choice_map[label] for label in valid_labels)

    # ── build the prompt ──────────────────────────────────────────────────────
    batch_structured_prompt = """
{question_text}

Options:
{options_text}

Return your answer as JSON with this exact shape:
{{"choice": "A"}}
Use only one label from the provided options.
"""

    batch_messages = [
        {"role": "system", "content": "You are a helpful tutor. Return only valid JSON with one field named choice."},
        {"role": "user",   "content": batch_structured_prompt.format(question_text=full_question_text, options_text=options_text)}
    ]

    # ── call the model ────────────────────────────────────────────────────────
    batch_response = model.create_chat_completion(
        messages=batch_messages,
        response_format={
            "type": "json_object",
            "schema": {
                "type": "object",
                "properties": {
                    "choice": {"type": "string", "enum": valid_labels}
                },
                "required": ["choice"]
            }
        },
        temperature=0.0
    )

    # ── parse the response ────────────────────────────────────────────────────
    batch_raw_text   = batch_response["choices"][0]["message"]["content"]
    extracted_answer = ""

    try:
        batch_json       = json.loads(batch_raw_text)
        extracted_answer = str(batch_json.get("choice", "")).strip().upper()
    except Exception:
        extracted_answer = ""

    # fallback: scan raw text for a valid label if JSON parsing failed
    if extracted_answer not in valid_labels:
        fallback_pattern = r"\b(" + "|".join(valid_labels) + r")\b"
        fallback_match   = re.search(fallback_pattern, batch_raw_text.upper())
        if fallback_match is not None:
            extracted_answer = fallback_match.group(1)

    # ── check the answer ──────────────────────────────────────────────────────
    correct_answer_text = str(practice_answers[item_index]).strip().upper()
    is_correct = (
        len(correct_answer_text) > 0
        and len(extracted_answer) > 0
        and extracted_answer == correct_answer_text[0]
    )

    print(f"Q{item_index + 1}: model={extracted_answer}  correct={correct_answer_text[0]}  {'✓' if is_correct else '✗'}")

    batch_results.append({
        "Section":        chosen_section,
        "Difficulty":     chosen_difficulty,
        "Question":       practice_questions[item_index],
        "Correct Answer": practice_answers[item_index],
        "Model Guess":    extracted_answer,
        "Correct?":       is_correct
    })

# display final results table
batch_results_table = pd.DataFrame(batch_results)
batch_results_table

Q1: model=A  correct=B  ✗
Q2: model=A  correct=B  ✗
Q3: model=B  correct=B  ✓
Q4: model=A  correct=B  ✗


,Section,Difficulty,Question,Correct Answer,Model Guess,Correct?
0,Math,Medium,Which of the following is equivalent to (x+y)^...,B,A,False
1,Math,Medium,"If 2x + 3y = 12 and x - y = 1, what is the val...",B,A,False
2,Math,Medium,A circle has a radius of 5 units. What is the ...,B,B,True
3,Math,Medium,A circle has a radius of 5 units. What is the...,B,A,False


In [42]:
# review a sample of results — shows question, model's guess, and correct answer side by side
sample_size = min(3, len(batch_results_table))
batch_results_table[:sample_size][["Question", "Model Guess", "Correct Answer"]]

,Question,Model Guess,Correct Answer
0,Which of the following is equivalent to (x+y)^...,A,B
1,"If 2x + 3y = 12 and x - y = 1, what is the val...",A,B
2,A circle has a radius of 5 units. What is the ...,B,B


Checkpoint #3: Try an easy question from a subject you choose. Did the model get it right? Explain how you checked.

In [44]:
# checkpoint #3 — reflection on batch results
checkpoint3_widget = widgets.RadioButtons(
    options=[
        "A: Yes, the model got it right and I verified by checking the answer key",
        "B: No, the model got it wrong — the answer key showed a different letter",
        "C: I'm not sure how to check",
    ],
    description="Your answer:",
    layout=widgets.Layout(width="600px")
)
submit_btn3 = widgets.Button(description="Submit", button_style="success")
out3 = widgets.Output()

def on_submit3(_):
    with out3:
        clear_output()
        selected = checkpoint3_widget.value[0]
        if selected == "A":
            print("✓ Great — you verified the answer correctly!")
        elif selected == "B":
            print("✓ Good observation! Check the 'Correct Answer' column in the results table.")
        else:
            print("Hint: compare 'Model Guess' and 'Correct Answer' in the results table above.")
        with open('answers.txt', 'a') as f:
            f.write("Checkpoint 3: " + checkpoint3_widget.value + "\n")

submit_btn3.on_click(on_submit3)
display(checkpoint3_widget, submit_btn3, out3)

RadioButtons(description='Your answer:', layout=Layout(width='600px'), options=('A: Yes, the model got it righ…

Button(button_style='success', description='Submit', style=ButtonStyle())

Output()

Print a summary showing how many questions the model got right overall.

In [46]:
# count how many questions the model answered correctly
correct_count = sum(1 for row in batch_results if row["Correct?"])
accuracy      = correct_count / len(batch_results) * 100

# build result rows
result_rows = "".join(
    f"""<tr>
        <td style='padding:8px 14px;'>{row['Question'][:60]}...</td>
        <td style='padding:8px 14px; text-align:center;'>{row['Correct Answer']}</td>
        <td style='padding:8px 14px; text-align:center;'>{row['Model Guess']}</td>
        <td style='padding:8px 14px; text-align:center;'>{'✓' if row['Correct?'] else '✗'}</td>
    </tr>"""
    for row in batch_results
)

display(HTML(f"""
<div style="font-family:sans-serif; font-size:13px; max-width:700px;">
  <div style="margin-bottom:12px;">
    <span style="font-size:16px; font-weight:bold;">Score: {correct_count} / {len(batch_results)}</span>
    &nbsp;&nbsp;
    <span style="color:#54a24b; font-weight:bold;">{accuracy:.0f}% accuracy</span>
  </div>
  <table style="border-collapse:collapse; width:100%; border:1px solid #d0d7de; border-radius:8px; overflow:hidden;">
    <tr style="background:#f0f0f0;">
      <th style="padding:8px 14px; text-align:left;">Question</th>
      <th style="padding:8px 14px; text-align:center;">Correct</th>
      <th style="padding:8px 14px; text-align:center;">Model</th>
      <th style="padding:8px 14px; text-align:center;">Result</th>
    </tr>
    {result_rows}
  </table>
</div>
"""))

Question,Correct,Model,Result
Which of the following is equivalent to (x+y)^2 - (x-y)^2?...,B,A,✗
"If 2x + 3y = 12 and x - y = 1, what is the value of x?...",B,A,✗
A circle has a radius of 5 units. What is the circumference ...,B,B,✓
A circle has a radius of 5 units. What is the circumference...,B,A,✗


## Summary

In this notebook you:
- Located a shared model directory and selected a GGUF model
- Loaded the model with `llama-cpp-python`
- Tested it on individual SAT questions from both English and Math sections
- Built a mini practice set filtered by difficulty and subject
- Ran a batch evaluation and compared the model's answers to the answer key

**What the results tell us:**
Small models like Qwen2-1.5B struggle with Medium and Hard math questions.
Try switching to Easy difficulty or English section to see better performance.
Your reflections are saved in `answers.txt`.